# (노트) 텐서보드 사용방법1

- toc:true
- branch: master
- badges: true
- comments: true
- author: 신록예찬
- hide: false
- categories: [데이터과학]

### imports

In [2]:
import tensorflow as tf

In [3]:
import numpy as np

In [4]:
import matplotlib.pyplot as plt

### data

In [5]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()

In [6]:
X = x_train.reshape(-1,28,28,1)/255
y = tf.keras.utils.to_categorical(y_train)
XX = x_test.reshape(-1,28,28,1)/255
yy = tf.keras.utils.to_categorical(y_test)

### 텐서보드 기본시각화 

In [6]:
net = tf.keras.Sequential()
net.add(tf.keras.layers.Flatten()) 
net.add(tf.keras.layers.Dense(50,activation='relu')) 
net.add(tf.keras.layers.Dense(10,activation='softmax')) 
net.compile(optimizer='adam',loss=tf.losses.categorical_crossentropy,metrics='accuracy')

2022-05-23 14:43:39.829725: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:939] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero


In [82]:
#collapse
!rm -rf logs
cb1 = tf.keras.callbacks.TensorBoard()
net.fit(X,y,epochs=200,batch_size=200,validation_split=0.2,callbacks=cb1)

Epoch 1/200
240/240 [==============================] - 0s 1ms/step - loss: 0.6966 - accuracy: 0.7706 - val_loss: 0.4897 - val_accuracy: 0.8365
Epoch 2/200
240/240 [==============================] - 0s 915us/step - loss: 0.4602 - accuracy: 0.8431 - val_loss: 0.4603 - val_accuracy: 0.8413
Epoch 3/200
240/240 [==============================] - 0s 908us/step - loss: 0.4161 - accuracy: 0.8560 - val_loss: 0.4279 - val_accuracy: 0.8543
Epoch 4/200
240/240 [==============================] - 0s 925us/step - loss: 0.3944 - accuracy: 0.8634 - val_loss: 0.4000 - val_accuracy: 0.8611
Epoch 5/200
240/240 [==============================] - 0s 927us/step - loss: 0.3737 - accuracy: 0.8705 - val_loss: 0.3903 - val_accuracy: 0.8638
Epoch 6/200
240/240 [==============================] - 0s 905us/step - loss: 0.3609 - accuracy: 0.8743 - val_loss: 0.3827 - val_accuracy: 0.8669
Epoch 7/200
240/240 [==============================] - 0s 919us/step - loss: 0.3491 - accuracy: 0.8779 - val_loss: 0.3850 - val_accu

`-` 파이썬 3.10의 경우 아래의 수정이 필요

`?/python3.10/site-packages/tensorboard/_vendor/html5lib/_trie/_base.py` 을 열고
```python
from collections import Mapping ### 수정전
from collections.abc import Mapping ### 수정후 
```
와 같이 수정한다. 

- 에러나는이유? 파이썬 3.10부터는 `from collections.abc import Mapping`이 실행되고 `from collections import Mapping`은 실행안된다. 

`-` 텐서보드 여는 방법1

In [1]:
%load_ext tensorboard
#%tensorboard --logdir logs --host 0.0.0.0

- 노트북에서 바로보인다..

`-` 텐서보드 여는 방법2 

In [85]:
!tensorboard --logdir logs --host 0.0.0.0

2022-05-22 21:19:44.992159: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:939] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2022-05-22 21:19:45.013209: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:939] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2022-05-22 21:19:45.013333: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:939] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero

NOTE: Using experimental fast data loading logic. To disable, pass
    "--load_fast=false" and report issues on GitHub. More details:
    https://github.com/tensorflow/tensorboard/issues/4784

TensorBoard 2.6.0 at http://0.0.0.0:6007/ (Press CTRL+C to quit)
^C


- 이제 `210.117.173.184:6007` 로 가면된다. 

### 텐서보드에서 그래프 보는 방법

`-` 그래프에서 tag를 keras로 변경하면 우리의 아키텐처가 간단하게 보인다. (지금은 이게 너무 간단하여 쓸모없어 보인다) 

### 조기종료 

`-` 텐서보드를 살펴보니 50에폭 이후로는 과적합이 진행되는 듯 하다. -> 50번까지만 학습한 모형이 있다면 그게 젤 좋은 것 같다. 

In [88]:
cb2 = tf.keras.callbacks.EarlyStopping() # 조기종료 콜백생성 

`-` 이미학습한 네트워크를 초기화 

In [87]:
tf.random.set_seed(43052)
net = tf.keras.Sequential()
net.add(tf.keras.layers.Flatten()) 
net.add(tf.keras.layers.Dense(50,activation='relu')) 
net.add(tf.keras.layers.Dense(10,activation='softmax')) 
net.compile(optimizer='adam',loss=tf.losses.categorical_crossentropy,metrics='accuracy')
net.fit(X,y,epochs=200,batch_size=200,validation_split=0.2,callbacks=cb2)

Epoch 1/200
1/1 [==============================] - 0s 464ms/step - loss: 2.3854 - accuracy: 0.1111 - val_loss: 2.1957 - val_accuracy: 0.1962
Epoch 2/200
1/1 [==============================] - 0s 44ms/step - loss: 2.1992 - accuracy: 0.1976 - val_loss: 2.0640 - val_accuracy: 0.2648


- 2번 돌고 멈추었다? 

In [91]:
cb2 = tf.keras.callbacks.EarlyStopping(patience=1) # 조기종료 콜백생성 - val_loss가 증가하는 순간 멈춘다. 

In [92]:
tf.random.set_seed(43052)
net = tf.keras.Sequential()
net.add(tf.keras.layers.Flatten()) 
net.add(tf.keras.layers.Dense(50,activation='relu')) 
net.add(tf.keras.layers.Dense(10,activation='softmax')) 
net.compile(optimizer='adam',loss=tf.losses.categorical_crossentropy,metrics='accuracy')
net.fit(X,y,epochs=200,batch_size=200,validation_split=0.2,callbacks=cb2)

Epoch 1/200
240/240 [==============================] - 0s 1ms/step - loss: 0.7184 - accuracy: 0.7581 - val_loss: 0.5077 - val_accuracy: 0.8276
Epoch 2/200
240/240 [==============================] - 0s 885us/step - loss: 0.4752 - accuracy: 0.8386 - val_loss: 0.4793 - val_accuracy: 0.8342
Epoch 3/200
240/240 [==============================] - 0s 889us/step - loss: 0.4304 - accuracy: 0.8517 - val_loss: 0.4386 - val_accuracy: 0.8497
Epoch 4/200
240/240 [==============================] - 0s 930us/step - loss: 0.4048 - accuracy: 0.8582 - val_loss: 0.4029 - val_accuracy: 0.8603
Epoch 5/200
240/240 [==============================] - 0s 857us/step - loss: 0.3832 - accuracy: 0.8669 - val_loss: 0.3932 - val_accuracy: 0.8619
Epoch 6/200
240/240 [==============================] - 0s 908us/step - loss: 0.3697 - accuracy: 0.8705 - val_loss: 0.3842 - val_accuracy: 0.8657
Epoch 7/200
240/240 [==============================] - 0s 927us/step - loss: 0.3569 - accuracy: 0.8759 - val_loss: 0.3844 - val_accu

In [93]:
cb2 = tf.keras.callbacks.EarlyStopping(patience=2) # 조기종료 콜백생성 - val_loss가 증가하는 순간 멈추는데 1번은 봐줌

In [94]:
tf.random.set_seed(43052)
net = tf.keras.Sequential()
net.add(tf.keras.layers.Flatten()) 
net.add(tf.keras.layers.Dense(1000,activation='relu')) 
net.add(tf.keras.layers.Dense(1000,activation='relu')) 
net.add(tf.keras.layers.Dense(10,activation='softmax')) 
net.compile(optimizer='adam',loss=tf.losses.categorical_crossentropy,metrics='accuracy')
net.fit(X,y,epochs=200,batch_size=60000,validation_split=0.2,callbacks=cb2,verbose=2)

Epoch 1/200
1/1 - 0s - loss: 2.3321 - accuracy: 0.1400 - val_loss: 1.6946 - val_accuracy: 0.4044 - 495ms/epoch - 495ms/step
Epoch 2/200
1/1 - 0s - loss: 1.6966 - accuracy: 0.4030 - val_loss: 1.2610 - val_accuracy: 0.6334 - 61ms/epoch - 61ms/step
Epoch 3/200
1/1 - 0s - loss: 1.2643 - accuracy: 0.6277 - val_loss: 1.0203 - val_accuracy: 0.6495 - 61ms/epoch - 61ms/step
Epoch 4/200
1/1 - 0s - loss: 1.0329 - accuracy: 0.6471 - val_loss: 0.8936 - val_accuracy: 0.6963 - 59ms/epoch - 59ms/step
Epoch 5/200
1/1 - 0s - loss: 0.9059 - accuracy: 0.6939 - val_loss: 0.8326 - val_accuracy: 0.6980 - 61ms/epoch - 61ms/step
Epoch 6/200
1/1 - 0s - loss: 0.8361 - accuracy: 0.6983 - val_loss: 0.7802 - val_accuracy: 0.7111 - 62ms/epoch - 62ms/step
Epoch 7/200
1/1 - 0s - loss: 0.7904 - accuracy: 0.7103 - val_loss: 0.7098 - val_accuracy: 0.7417 - 60ms/epoch - 60ms/step
Epoch 8/200
1/1 - 0s - loss: 0.7251 - accuracy: 0.7369 - val_loss: 0.6902 - val_accuracy: 0.7615 - 60ms/epoch - 60ms/step
Epoch 9/200
1/1 - 0s -

`-` 조기종료콜백과 텐서보드콜백을 같이 쓰고싶다면? 

In [95]:
tf.random.set_seed(43052)
net = tf.keras.Sequential()
net.add(tf.keras.layers.Flatten()) 
net.add(tf.keras.layers.Dense(50,activation='relu')) 
net.add(tf.keras.layers.Dense(10,activation='softmax')) 
net.compile(optimizer='adam',loss=tf.losses.categorical_crossentropy,metrics='accuracy')
net.fit(X,y,epochs=200,batch_size=200,validation_split=0.2,callbacks=[cb1,cb2],verbose=0)

In [31]:
#%tensorboard --logdir logs --host 0.0.0.0

### 하이퍼파라메터 튜닝

`-` 참고문헌

- https://www.tensorflow.org/tensorboard/hyperparameter_tuning_with_hparams

`-` 하이퍼파라메터 설정

In [8]:
from tensorboard.plugins.hparams import api as hp

In [11]:
tf.summary.experimental.get_step()

In [39]:
!rm -rf logs
for u in [200,2000]:
    for d in [0.0,0.8]: 
        for o in ['adam','sgd']:
            #hparams = {HP_NUM_UNITS: num_units, HP_DROPOUT: dropout_rate, HP_OPTIMIZER: optimizer,}
            logdir = 'logs/hparam_tuning_{}_{}_{}'.format(u,d,o)
            with tf.summary.create_file_writer(logdir).as_default():
                #hp.hparams(hparams)          
                net = tf.keras.Sequential()
                net.add(tf.keras.layers.Flatten())
                net.add(tf.keras.layers.Dense(u, activation='relu'))
                net.add(tf.keras.layers.Dropout(d))
                net.add(tf.keras.layers.Dense(10, activation='softmax'))
                net.compile(optimizer=o,loss=tf.losses.categorical_crossentropy,metrics=['accuracy','Recall'])
                cb3 = hp.KerasCallback(logdir, {'유닛수':u, '드랍아웃':d, '옵티마이저':o})
                net.fit(X,y, epochs=4, callbacks=cb3)
                _mymetric=sum(net.evaluate(XX,yy)[1:]) # net.evaluate(XX,yy)는 [loss,accuracy,recall]를 리턴함
                tf.summary.scalar('애큐러시+리콜', _mymetric, step=1) ## 내맘대로 메트릭을 만들어서 그거 찍어보고 싶다!

Epoch 1/4
1875/1875 [==============================] - 2s 1ms/step - loss: 0.4889 - accuracy: 0.8275 - recall: 0.7802
Epoch 2/4
1875/1875 [==============================] - 2s 1ms/step - loss: 0.3697 - accuracy: 0.8658 - recall: 0.8380
Epoch 3/4
1875/1875 [==============================] - 2s 1ms/step - loss: 0.3286 - accuracy: 0.8799 - recall: 0.8570
Epoch 4/4
313/313 [==============================] - 0s 923us/step - loss: 0.3504 - accuracy: 0.8761 - recall: 0.8551
Epoch 1/4
1875/1875 [==============================] - 2s 1ms/step - loss: 0.7340 - accuracy: 0.7631 - recall: 0.6015
Epoch 2/4
1875/1875 [==============================] - 2s 1ms/step - loss: 0.5103 - accuracy: 0.8259 - recall: 0.7598
Epoch 3/4
1875/1875 [==============================] - 2s 1ms/step - loss: 0.4655 - accuracy: 0.8389 - recall: 0.7892
Epoch 4/4
313/313 [==============================] - 0s 886us/step - loss: 0.4660 - accuracy: 0.8365 - recall: 0.7904
Epoch 1/4
1875/1875 [==============================] - 2

In [2]:
%load_ext tensorboard
#%tensorboard --logdir logs --host 0.0.0.0

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard
